# Identifying duplicates and checking number of not uniquely identified proteins in redundant search

# Setup

## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import re
import openpyxl
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/05_MaxQuant"):
    print("WARNING: The working directory is not set to the '05_MaxQuant'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '05_MaxQuant' folder (\"{cwd}\").")

In [ ]:
# Data directories
prenylation_dir = "../04_Prenylation_Database/data/"
mapping_dir = "../01_UniProt/data/mapping/"
isoform_database_dir = "../02_Isoform_Database/"
duplicates_dir = "../01_UniProt/data/sequence_duplicates/"

redundant_db_dir = "data/redundant_DB/"

total_extracts_dir = "data/ProteinGroups_LNf/te/"
prenylation_enrichment_dir = "data/ProteinGroups_LNf/clickit/"

## Read in MaxQuant Results and duplicates list

In [ ]:
isoform_fasta = read_fastafile(os.path.join(isoform_database_dir, 'Isoform_Database_only_iso.fasta'))
canonical_fasta = read_fastafile(os.path.join(isoform_database_dir, 'unique_canonical.fasta'))
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv')) 
duplicates_list = pd.read_csv(os.path.join(duplicates_dir, 'duplicates_list_01042026.csv'))

####### REDUNDANT DB #######
#total extracts
#protgroup_act_te = pd.read_excel(os.path.join(redundant_db_dir, 'proteinGroups_act_te_redundant.xlsx')) # Th1-Cells activated vs non-activated
#pep_act_te = pd.read_excel(os.path.join(redundant_db_dir, 'peptides_act_te_redundant.xlsx')) 
#protgroup_stat_te = pd.read_excel(os.path.join(redundant_db_dir, 'proteinGroups_stat_te_redundant.xlsx')) # activated Th1 cells +/- statin treatment
#peptides_stat_te = pd.read_excel(os.path.join(redundant_db_dir, 'peptides_stat_te_redundant.xlsx'))

#prenylation enriched (with clickit-reaction)
#protgroup_act_clickit = pd.read_excel(os.path.join(redundant_db_dir, 'proteinGroups_act_clickit_redundant.xlsx')) # Th1-Cells activated vs non-activated
#pep_act_clickit = pd.read_excel(os.path.join(redundant_db_dir, 'peptides_act_clickit_redundant.xlsx'))
#protgroup_stat_clickit = pd.read_excel(os.path.join(redundant_db_dir, 'proteinGroups_stat_clickit_redundant.xlsx')) # activated Th1 cells +/- statin treatment
#pep_stat_clickit = pd.read_excel(os.path.join(redundant_db_dir, 'peptides_stat_clickit_redundant.xlsx'))

####### NON-REDUNDANT DB #######
#total extracts
mq_results_act_te = pd.read_excel(os.path.join(total_extracts_dir, 'Prenyl_act_te_LNf_proteinGroups.xlsx')) # Th1-Cells activated vs non-activated
mq_results_stat_te = pd.read_excel(os.path.join(total_extracts_dir, 'Prenyl_stat_te_LNf_proteinGroups.xlsx')) # activated Th1 cells +/- statin treatment
mq_results_FTI_te = pd.read_excel(os.path.join(total_extracts_dir, 'Prenyl_FTI_te_LNf_proteinGroups.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitor

#prenylation enriched (with clickit-reaction)
mq_results_act_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'Prenyl_act_clickit_LNf_proteinGroups.xlsx')) # Th1-Cells activated vs non-activated
mq_results_stat_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'Prenyl_stat_clickit_LNf_proteinGroups.xlsx')) # activated Th1 cells +/- statin treatment
mq_results_FTI_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'Prenyl_FTI_clickit_LNf_proteinGroups.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitorredundant 


# Filter Fasta headers by duplicates fasta headers

In [ ]:
isoform_set = set(isoform_fasta['seqID'])
duplicates_set = set(duplicates_list['seqID'])

In [ ]:
def filter_maxquant(df, duplicates_set,
                             id_col='Protein IDs', 
                             header_col='Fasta headers'):
    """
    Cleans MaxQuant results and splits them into Isoforms and Prenylation variants
    using statistical thresholds for FDR and peptide count.
    """
    # Basic Cleaning (Decoys, Contaminants, and Only identified by site)
    initial_count = len(df)
    
    # Remove rows based on MaxQuant flag columns
    flags = ['Reverse', 'Potential contaminant', 'Only identified by site']
    for col in flags:
        if col in df.columns:
            df = df[df[col] != '+']
    

    clean_df = df.copy()
    
    # 3. Helper function for set intersection
    def check_match(header_str, target_set):
        if pd.isna(header_str): return False
        # Split by semicolon as MaxQuant often concatenates IDs
        return any(h.strip() in target_set for h in str(header_str).split(';'))

    # 4. Create the two filtered DataFrames
    df_duplicates = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, duplicates_set))
    ].copy()


    # Reporting
    print(f"--- Filter Report ---")
    print(f"Initial Rows: {initial_count}")
    print(f"Duplicates hits: {len(df_duplicates)}")
    
    return df_duplicates

In [ ]:
# Execution:
#total extracts
duplicates_act_te = filter_maxquant(protgroup_act_te, duplicates_set)
duplicates_stat_te = filter_maxquant(protgroup_stat_te, duplicates_set)

#prentylation enrichment
duplicates_act_clickit = filter_maxquant(protgroup_act_clickit, duplicates_set)
duplicates_stat_clickit = filter_maxquant(protgroup_stat_clickit, duplicates_set)

In [ ]:
# Save splice variants as csv 
duplicates_act_te.to_csv(os.path.join(redundant_db_dir, 'duplicates_act_te.csv'), index=False)
duplicates_stat_te.to_csv(os.path.join(redundant_db_dir, 'duplicates_stat_te.csv'), index=False)

duplicates_act_clickit.to_csv(os.path.join(redundant_db_dir, 'duplicates_act_clickit.csv'), index=False)
duplicates_stat_clickit.to_csv(os.path.join(redundant_db_dir, 'duplicates_stat_clickit.csv'), index=False)

In [ ]:
duplicates_act_clickit

# Number of uniquely identified proteins 

In [ ]:
def unique_proteins(df):
    """
    Analyzes a MaxQuant protein groups pandas DataFrame to find the number 
    and percentage of unique proteins vs. multi-protein groups after filtering.
    """
    # 1. Standard MaxQuant filtering columns
    # We create a filter mask. If a column doesn't exist in your df, it skips it.
    filter_cond = pd.Series(True, index=df.index)
    
    flag_cols = ['Only identified by site', 'Reverse', 'Potential contaminant']
    for col in flag_cols:
        if col in df.columns:
            # Keep rows that are NOT flagged with '+'
            filter_cond &= (df[col] != '+')
            
    df_filtered = df[filter_cond].copy()
    total_valid_entries = len(df_filtered)
    
    if total_valid_entries == 0:
        print("No valid proteins found after filtering. Check your dataframe columns.")
        return None

    # 2. Check for the Protein IDs column
    if 'Protein IDs' not in df_filtered.columns:
        raise KeyError("The DataFrame must contain a 'Protein IDs' column.")

    # 3. Count how many protein IDs are in each row (separated by ';')
    df_filtered['Protein_Count'] = df_filtered['Protein IDs'].apply(
        lambda x: len(str(x).split(';')) if pd.notna(x) else 0
    )
    
    # 4. Separate into Uniquely Identified (1 ID) vs. Multi-Protein Groups (>= 2 IDs)
    num_unique = len(df_filtered[df_filtered['Protein_Count'] == 1])
    num_multi = len(df_filtered[df_filtered['Protein_Count'] > 1])
    
    # 5. Calculate percentages
    pct_unique = (num_unique / total_valid_entries) * 100
    pct_multi = (num_multi / total_valid_entries) * 100
    
    # 6. Print Results Summary
    print("## MaxQuant Protein Groups Analysis Summary")
    print(f"* Total Valid Protein Groups (Post-filtering): {total_valid_entries}")
    print(f"* Uniquely Identified Proteins (1 protein/group): {num_unique} ({pct_unique:.2f}%)")
    print(f"* Multi-Protein Groups (≥ 2 proteins/group): {num_multi} ({pct_multi:.2f}%)")
    print("-----------------------------------------------------------------------------")
    
    return {
        "total_valid_groups": total_valid_entries,
        "uniquely_identified_proteins": num_unique,
        "multi_protein_groups": num_multi,
        "pct_unique": pct_unique,
        "pct_multi": pct_multi
    }

In [ ]:
# Execution:
#total extracts redundant
groups_vs_unique_act_te = unique_proteins(protgroup_act_te)
groups_vs_unique_stat_te = unique_proteins(protgroup_stat_te)

#prentylation enrichment redundant
groups_vs_unique_act_clickit = unique_proteins(protgroup_act_clickit)
groups_vs_unique_stat_clickit = unique_proteins(protgroup_stat_clickit)

#te non-renundant
groups_vs_unique_act_te_nr = unique_proteins(mq_results_act_te)
groups_vs_unique_stat_te_nr = unique_proteins(mq_results_stat_te)

#clickit non-redundant
groups_vs_unique_act_clickit_nr = unique_proteins(mq_results_act_clickit)
groups_vs_unique_stat_clickit_nr = unique_proteins(mq_results_stat_clickit)